# Ireland Electricity Generation Analytics
## Project 04 — How Ireland changed what powers its electricity system

**Research question**

> How did Ireland transform its electricity-generation system between 2010 and 2025, and what does that transition mean for generation mix, operating flexibility, thermal efficiency, imported-fuel exposure, carbon cost and generation economics?

This notebook separates observed data, derived metrics and interpretation. It does not treat correlation as causation or compare unlike cost concepts as though they were identical.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO = "danielsantos2310/Ireland-Electricity-Imports-Analytics."
BRANCH = "main"
REMOTE_DATA = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/processed"

REQUIRED_FILE = "generation_annual_summary_2010_2025.csv"

def find_local_data_dir():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "processed",
        cwd.parent / "data" / "processed",
        Path("../data/processed").resolve(),
    ]
    for candidate in candidates:
        if (candidate / REQUIRED_FILE).exists():
            return candidate
    return None

DATA = find_local_data_dir()

def load_csv(filename):
    """Load locally when available; otherwise read the published GitHub dataset.

    This supports repository-based Jupyter/VS Code runs and direct GitHub→Colab
    use, where the rest of the repository is not cloned automatically.
    """
    if DATA is not None:
        return pd.read_csv(DATA / filename)
    return pd.read_csv(f"{REMOTE_DATA}/{filename}")

CHARTS = (Path.cwd().parent / "charts") if Path.cwd().name == "notebooks" else (Path.cwd() / "charts")
CHARTS.mkdir(parents=True, exist_ok=True)

print(f"Data source: {DATA if DATA is not None else REMOTE_DATA}")


## 1. Load canonical datasets


In [ ]:
annual = load_csv('generation_annual_summary_2010_2025.csv')
annual_long = load_csv('generation_annual_long_2010_2025.csv')
monthly = load_csv('generation_monthly_2010_2025.csv')
cso_2025 = load_csv('cso_supply_monthly_2025.csv')
thermal = load_csv('thermal_generation_efficiency_2010_2025.csv')
fuel_supply = load_csv('fuel_supply_dependency_2010_2025.csv')
validation = load_csv('generation_source_validation_2023_2025.csv')
concentration = load_csv('generation_concentration_2010_2025.csv')
relationships = load_csv('monthly_operating_relationships_by_year.csv')
seasonality = load_csv('seasonality_profile_2021_2025.csv')
ops_2025 = load_csv('monthly_grid_operating_metrics_2025.csv')
ress = load_csv('ress_auction_price_history_2020_2025.csv')
gas_cost = load_csv('gas_fuel_carbon_cost_sensitivity_2025.csv')
evidence = load_csv('project04_evidence_hierarchy.csv')

print(f"Loaded {len(annual)} annual rows, {len(monthly)} monthly generation rows, and {len(evidence)} evidence statements.")


In [ ]:
required_years = set(range(2010, 2026))
assert set(annual['year']) == required_years, "Annual summary must cover every year from 2010 through 2025."
assert set(thermal['year']) == required_years, "Thermal-efficiency data must cover 2010–2025."
assert monthly['year'].min() == 2010 and monthly['year'].max() == 2025, "Monthly generation coverage is incomplete."
assert {'Gas', 'Wind', 'Coal', 'Solar'}.issubset(set(annual_long['source'])), "Core generation sources are missing."
assert len(ops_2025) == 12, "2025 operating table must contain 12 months."
assert abs(float(annual.loc[annual['year'].eq(2025), 'gas_plus_wind_share_pct'].iloc[0]) - 87.6404) < 0.01
print("Data integrity checks passed: 2010–2025 coverage and core Project 04 datasets are present.")


## 2. Source reconciliation
SEAI monthly generation aggregates are checked against CSO annual grid-scale generation for 2023–2025. The series reconcile extremely closely, supporting use of the SEAI export as the 2010–2025 backbone while retaining CSO as the recent operational/validation layer.


In [ ]:
validation.groupby('year').agg(
    max_abs_pct_difference=('difference_pct_of_cso', lambda s: s.abs().max()),
    mean_abs_pct_difference=('difference_pct_of_cso', lambda s: s.abs().mean())
)


## 3. Structural transition — generation changed composition more than scale


In [ ]:
a = annual.set_index('year')
summary = pd.DataFrame({
    '2010': [a.loc[2010,'total_generation_gwh'], a.loc[2010,'gas_share_pct'], a.loc[2010,'wind_share_pct'], a.loc[2010,'coal_share_pct']],
    '2025': [a.loc[2025,'total_generation_gwh'], a.loc[2025,'gas_share_pct'], a.loc[2025,'wind_share_pct'], a.loc[2025,'coal_share_pct']],
}, index=['Total generation GWh','Gas share %','Wind share %','Coal share %'])
summary


In [ ]:
mix = annual_long.pivot(index='year',columns='source',values='generation_gwh').fillna(0)
mix[['Gas','Wind','Coal','Peat & Biomass','Solar','Hydro']].plot(figsize=(12,7), marker='o')
plt.title('Ireland domestic electricity generation by major source, 2010–2025')
plt.ylabel('GWh')
plt.show()


## 4. Concentration — the system increasingly revolves around its two largest sources


In [ ]:
concentration[['year','top_1_source','top_1_share_pct','top_2_source','top_2_share_pct','top_2_combined_share_pct','effective_number_of_equal_sources']].tail(8)


## 5. Operating relationship — wind and gas are not a simple one-for-one seesaw
Monthly gas–wind correlation varies substantially by year. Negative correlation in some years is consistent with gas reducing output when wind is high, but correlation alone does not identify balancing causally; demand, imports, outages, storage, prices and other sources also matter.


In [ ]:
relationships[['year','gas_wind_corr','wind_solar_corr','gas_monthly_cv','wind_monthly_cv']].tail(10)


## 6. Seasonality — wind and solar have different profiles


In [ ]:
seasonality[['month','Gas_mean','Wind_mean','Solar_mean']]


## 7. Recent grid supply — imports and storage


In [ ]:
ops_2025[['period','generation_gwh','net_imports_gwh','storage_discharge_gwh','net_import_share_proxy_pct']].round(2)


## 8. Thermal generation — less fuel, better efficiency, more gas concentration


In [ ]:
thermal[['year','public_thermal_input_ktoe','public_thermal_electricity_output_ktoe','public_thermal_efficiency_pct','gas_input_share_pct','coal_input_share_pct','peat_input_share_pct']].tail(10)


## 9. Fuel origin — domestic generation is not the same as indigenous energy


In [ ]:
fuel_supply.query('year == 2025')[['fuel','indigenous_production_ktoe','imports_ktoe','primary_energy_requirement_ktoe','import_share_of_indigenous_plus_imports_pct']].sort_values('imports_ktoe',ascending=False)


## 10. Renewable investment benchmarks — RESS history
RESS prices are successful-auction revenue/support benchmarks. They are not dispatch costs and should not be interpreted as plant marginal cost or a universal LCOE.


In [ ]:
ress[['auction','auction_year','all_projects_eur_mwh','wind_eur_mwh','solar_eur_mwh']]


## 11. Gas economics — sensitivity, not false precision
The gas model uses:
- SEAI 2025 provisional natural-gas combustion factor: 203.5 gCO₂/kWh fuel (NCV).
- 2025 EU ETS average auction price: €73.43/tCO₂.
- A range of gas fuel prices and thermal efficiencies.

It excludes plant-specific O&M, startup costs, capacity payments, financing, hedging and network charges.


In [ ]:
gas_cost.query('efficiency_pct == 50')


## 12. Evidence hierarchy


In [ ]:
evidence


## Reproducibility and source notes

The repository keeps original-source data separate from analytical outputs. The public release includes the processed datasets required to reproduce the figures in this notebook.

Primary evidence:
- Central Statistics Office (CSO), *Electricity Supply by Source*.
- Sustainable Energy Authority of Ireland (SEAI), electricity-generation statistics, National Energy Balance and Energy by Fuel.
- EirGrid, RESS final auction results.
- European Commission, EU ETS auction reporting.
- Gas Networks Ireland, imbalance-price / SAP documentation.

Important cost interpretation:
- RESS strike prices are auction/support revenue benchmarks.
- The gas model is a fuel + carbon sensitivity, not a claim about any plant's private procurement or total LCOE.
- Gas Networks Ireland's SAP/imbalance framework and the official ONS/National Gas Transmission SAP series are used as market context, not as a substitute for plant-specific contracts.


## 13. Conclusion

Ireland's electricity transition between 2010 and 2025 was primarily a transformation in **composition**, not a dramatic expansion of domestic generation.

Wind expanded from a secondary source into one of the two dominant sources. Coal and peat retreated sharply, solar began scaling, and public thermal generation became smaller and more efficient.

Yet the transition did not eliminate external exposure. Gas remains the dominant thermal fuel and is mostly imported. Direct electricity imports also rose sharply between 2023 and 2025. The system is therefore increasingly built around a combination of **indigenous variable renewables, gas flexibility, interconnection and emerging storage**.

The most defensible economic comparison is not a single 'cost per MWh' league table. Gas economics are highly sensitive to fuel and carbon prices, while RESS prices are long-term support/revenue benchmarks for renewable investment. The distinction is essential.

This naturally leads to Project 05: **what electricity does Ireland import, where does it come from, and what does it cost?**
